In [ ]:
# 모! 듈! 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 통계분석용
from scipy import stats # 어노바
from statsmodels.stats.multicomp import pairwise_tukeyhsd # 어노바 친구 튜키
from scipy.stats import chi2_contingency # 그리고 Chi-square
from scipy.stats import shapiro # 정규성 검정용
from scipy.stats import probplot # 큐큐플롯
from scipy.stats import levene # 레벤 검정용 

import statsmodels.api as sm
from statsmodels.formula.api import ols

from sklearn.decomposition import PCA # 주성분분석
from sklearn.preprocessing import StandardScaler # 을 하려면 스케일러가 필요합니다 
from sklearn.feature_selection import mutual_info_classif # MI (결과는 다른 분 걸 사용함)
from sklearn.preprocessing import LabelEncoder 

import prince # FAMD용

# 그래프 기본 테마 설정
sns.set_theme(palette="Blues", style="whitegrid", font_scale=1) # 블루톤

# 그래프를 그리기 위한 기본 설정
plt.rcParams['font.family'] = 'Pretendard' # 프리텐다드
# plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['figure.figsize'] = 12, 6
plt.rcParams['font.size'] = 14 # 기본 폰트사이즈 (자세한 양식은 하기)
plt.rcParams['axes.unicode_minus'] = False

# 그래프 공통 적용 사항
# 1. 테마: 커스텀 팔레트('#5ca0de','#9dadbc')/히트맵: Blues
# 2. Font: 프리텐다드
# 3. 제목/라벨 크기: 16, 14(별도 설정이 있는 경우 12)
# 4. 제목 라벨: y = 1.03

# 읽기 전에
1. 통계분석 페이지입니다. 단, PPT에 있는 통계분석은 저 말고 다른 분도 같이 하신 결과입니다. 
2. 개인 깃헙에는 출력 정리 버전이 올라갈 예정입니다. 
3. 주석에 숫자가 달려있거나 파라미터가 세로로 나열된 경우(혹은 탭이 뻘건색인 경우): 제미나이가 도와줬습니다. 제미나이랑 저랑 스타일이 안 맞는 부분이 좀 있어서 차이가 있습니다. 출력 문구도 다 제미나이 작품입니다. 
4. 통계분석은 순서대로...가 아니라 같은 통계분석끼리 묶여있습니다. (단, 어노바-튜키는 한 그룹으로 간주)

In [ ]:
# 일단 불러야 뭘 한다
bank_df = pd.read_csv('data/bank_preprocess.csv')

# 저장하면서 뭔가 잘못된 것 같습니다. 
# 이거 개인적으로 캐글 EDA 할 때도 이랬습니다. 
bank_df.drop('Unnamed: 0', axis=1, inplace=True) 

bank_df

In [ ]:
bank_df.columns

# Chi-square
- 주로 두 범주형 변수 간의 연관성(독립성)을 분석하거나 관측된 데이터가 특정 분포에 얼마나 잘 맞는지를 평가하는 통계적 방법입니다. 
    - 예: 카이스트 공대, 포항공대 공과대학, 한양대 공대의 남녀 성비가 다른가? 
- deposit이 범주형이기때문에 ANOVA가 불가능해서 카이제곱 검정을 사용했습니다. (이후 카이스퀘어라고 쓰여있는 게 보이신다면 아, 카이제곱 했구나 하시면 됩니다)

## 연령대별 가입비

In [ ]:
# 20~50대/other(10대/60대 이상)
main_age_group = [20, 30, 40, 50] # 2, 3, 4, 50대
bank_df['group_category'] = bank_df['age_group'].apply(lambda x: f"{int(x)}대" if x in main_age_group else '기타(Others)') 

In [ ]:
# 교차표(Contingency Table) 생성
age_deposit_table = pd.crosstab(bank_df['group_category'], bank_df['deposit'])

print("--- 연령대별 가입 빈도 교차표 ---")
display(age_deposit_table)

# 카이제곱 검정
# 뭔지 잘 모르시겠다고요? 강사님이 찰떡설명 해주실거니까 수업할 때 집중!! 하시면 됩니다. 
chi2, p_val, dof, expected = chi2_contingency(age_deposit_table)

# 실제로 관측된 값이, ‘연령대와 가입 여부가 서로 무관하다’고 가정했을 때의 값에서 얼마나 크게 벗어났는지를 수치로 나타낸 것
print(f"\n카이제곱 통계량: {chi2:.2f}") 
print(f"p-value: {p_val}") # P-value (그 유의수준보다 작게 나와야 하는 값)
print(f"자유도: {dof}") # 아래 표는 연령대 카테고리가 5개, deposit이 2개(예스 노)라 4가 나옵니다. (5-1 * 2-1 해서 4)

# 결과 해석
# 사실 이것도 제미나이가 만들어줬어요.. 
alpha = 0.05
if p_val < alpha:
    print(f"\n해석: p-value({p_val:.4e})가 0.05보다 작으므로, 연령대에 따른 가입 여부의 차이는 통계적으로 유의미합니다.")
else:
    print(f"\n해석: p-value({p_val:.4f})가 0.05보다 크므로, 연령대와 가입 여부는 독립적(차이 없음)이라고 볼 수 있습니다.")

In [ ]:
# 카이제곱 표(?)
cont_table = pd.crosstab(bank_df['group_category'], bank_df['deposit'])
chi2, p, dof, expected = chi2_contingency(cont_table)

# 기대값을 데이터프레임으로 변환
expected_df = pd.DataFrame(expected, index=cont_table.index, columns=cont_table.columns)

# (실제값 - 기대값) 계산: 양수면 예상보다 많이 가입한 것, 음수면 적게 가입한 것
# 보시면 3050은 음수입니다. 
diff = cont_table - expected_df

print("--- 연령대별 가입(yes) 기여도 (실제 - 기대) ---")
display(diff[['yes']].sort_values(by='yes', ascending=False))

In [ ]:
# 시각화(PPT에는 안씀)
# 행별로 합계가 1(100%)이 되도록 정규화
age_deposit_ratio = pd.crosstab(bank_df['group_category'], bank_df['deposit'], normalize='index')

# 시각화
ax = age_deposit_ratio.plot(kind='bar', stacked=True, figsize=(10, 6), color=['#d9534f', '#5cb85c'])

plt.title('Deposit Success Rate by Age Group', fontsize=15)
plt.xlabel('Age Group')
plt.ylabel('Proportion (100%)')
plt.xticks(rotation=0)
plt.legend(title='Deposit', bbox_to_anchor=(1, 1))

# 비율 수치 표시
for p in ax.patches:
    width, height = p.get_width(), p.get_height()
    x, y = p.get_xy() 
    if height > 0: # 0인 경우 표시 안 함
        ax.text(x + width/2, y + height/2, f'{height:.1%}', 
                ha='center', va='center', fontweight='bold', color='white')

plt.tight_layout()
plt.show()

### 10대도 분리해서 확인해 본 결과

In [ ]:
# 10~50대/other(60대 이상)
main_age_group = [10,20, 30, 40, 50] # 2, 3, 4, 50대
bank_df['group_category'] = bank_df['age_group'].apply(lambda x: f"{int(x)}대" if x in main_age_group else '기타(Others)') 

# 교차표(Contingency Table) 생성
age_deposit_table = pd.crosstab(bank_df['group_category'], bank_df['deposit'])

print("--- 연령대별 가입 빈도 교차표 ---")
display(age_deposit_table)

# 카이제곱 검정
chi2, p_val, dof, expected = chi2_contingency(age_deposit_table)

print(f"\n카이제곱 통계량: {chi2:.6f}")
print(f"p-value: {p_val:.6f}")

# 결과 해석
alpha = 0.05
if p_val < alpha:
    print(f"\n해석: p-value({p_val:.4e})가 0.05보다 작으므로, 연령대에 따른 가입 여부의 차이는 통계적으로 유의미합니다.")
else:
    print(f"\n해석: p-value({p_val:.4f})가 0.05보다 크므로, 연령대와 가입 여부는 독립적(차이 없음)이라고 볼 수 있습니다.")

In [ ]:
cont_table = pd.crosstab(bank_df['group_category'], bank_df['deposit'])
chi2, p, dof, expected = chi2_contingency(cont_table)

# 기대값을 데이터프레임으로 변환
expected_df = pd.DataFrame(expected, index=cont_table.index, columns=cont_table.columns)

# (실제값 - 기대값) 계산: 양수면 예상보다 많이 가입한 것, 음수면 적게 가입한 것
diff = cont_table - expected_df

print("--- 연령대별 가입(yes) 기여도 (실제 - 기대) ---")
display(diff[['yes']].sort_values(by='yes', ascending=False))

In [ ]:
# 행별로 합계가 1(100%)이 되도록 정규화
age_deposit_ratio = pd.crosstab(bank_df['group_category'], bank_df['deposit'], normalize='index')

# 시각화
ax = age_deposit_ratio.plot(kind='bar', stacked=True, figsize=(10, 6), color=['#d9534f', '#5cb85c'])

plt.title('Deposit Success Rate by Age Group', fontsize=15)
plt.xlabel('Age Group')
plt.ylabel('Proportion (100%)')
plt.xticks(rotation=0)
plt.legend(title='Deposit', bbox_to_anchor=(1, 1))

# 비율 수치 표시
for p in ax.patches:
    width, height = p.get_width(), p.get_height()
    x, y = p.get_xy() 
    if height > 0: # 0인 경우 표시 안 함
        ax.text(x + width/2, y + height/2, f'{height:.1%}', 
                ha='center', va='center', fontweight='bold', color='white')

plt.tight_layout()
plt.show()

## 접촉 이력에 따른 전환률

In [ ]:
# 전처리 
# 결측치가 없는 깨끗한 데이터를 사용합니다. 근데 여기 결측값은 없어서... 
df_pure = bank_df.dropna(subset=['previous_group', 'deposit']).copy()

# 교차표(Contingency Table) 생성
# 행: 접촉 이력 그룹, 열: 가입 여부
cross_tab_pure = pd.crosstab(df_pure['previous_group'], df_pure['deposit'])

print("=== 전체 데이터 접촉 이력별 가입 빈도 ===")
display(cross_tab_pure)

# 카이제곱 검정 수행
chi2, p_value, dof, expected = chi2_contingency(cross_tab_pure)

print(f"\n카이제곱 통계량: {chi2:.2f}")
print(f"p-value: {p_value:.4g}") # 매우 작을 경우 지수 표기법 사용
print(f"자유도: {dof}")

# 결과 해석
if p_value < 0.05:
    print(f"\n[결론] p-value가 {p_value:.4g}로 유의수준 0.05보다 매우 작습니다.")
    print("접촉 이력과 가입 여부 사이에는 '매우 강력한' 통계적 상관관계가 있습니다.")
else:
    print(f"\n[결론] p-value가 {p_value:.4f}로 유의수준 0.05보다 큽니다.")
    print("접촉 이력과 가입 여부 사이에는 통계적으로 유의미한 관계가 발견되지 않았습니다.")

In [ ]:
# 교차표 생성 (과거 접촉 그룹별 가입 여부)
cross_tab_prev = pd.crosstab(bank_df['previous_group'], bank_df['deposit'])

# 카이제곱 검정 수행
chi2, p_value, dof, expected = chi2_contingency(cross_tab_prev)

# CVR 계산
cvr_prev = cross_tab_prev.copy()
cvr_prev['total'] = cvr_prev['no'] + cvr_prev['yes']
cvr_prev['CVR(%)'] = (cvr_prev['yes'] / cvr_prev['total']) * 100

# 결과 출력
print("=== 과거 접촉 이력(previous_group)별 CVR 분석 ===")
display(cvr_prev[['yes', 'total', 'CVR(%)']].sort_values(by='CVR(%)', ascending=False))

print(f"\n카이제곱 통계량: {chi2:.4f}")
print(f"p-value: {p_value:.4g}")
print(f"자유도: {dof}")

# 시각화
plt.figure(figsize=(10, 6))
ax = sns.barplot(x=cvr_prev.index, y=cvr_prev['CVR(%)'], order=cvr_prev['CVR(%)'].sort_values(ascending=False).index)

for p in ax.patches:
    ax.annotate(f'{p.get_height():.1f}%', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center', fontsize=11, color='black', xytext=(0, 7),
                textcoords='offset points')

plt.title("CVR by Previous Contact Group")
plt.ylabel("CVR (%)")
plt.show()

# 맨-휘트니 U 검정
- t-test랑 비슷...한건데 이제 데이터가 정규성을 갖지 않거나 표본 크기가 작을 때(30 미만) 씁니다. 
- 밑에 다시 한번 서술하겠지만, 사전 절차로 샤피로-윌크 검정과 QQplot(정규성), 레빈 검정(등분산성 확인)을 통해 정규성, 등분산성 확인 절차를 거쳤습니다. 
- 다 해놓고 생각해보니 표본 크기가 애초에 작아서 T-test를 못하네요. 

## 대출 유무(+잔고)에 따른 차이가 통계적으로 유의한가?

In [ ]:
# 카피떠서 진행 
bank_df1 = bank_df.query('balance_group == "plus"').copy()

# 대출 유무에 따라 나누는거기때문에 주담대+다른 대출이 없는 사람과 아닌 사람으로 나눌 예정입니다. 
def define_loan_group(row):
    if row['housing'] == 'no' and row['loan'] == 'no':
        return 'HNLN'
    else:
        return 'Has_Loan'

bank_df1['loan_group'] = bank_df1.apply(define_loan_group, axis=1)

# 잔고로 큐컷
bank_df1['balance_decile'] = pd.qcut(bank_df1['balance'], 10, labels=False)

# 그룹별 가입률 계산
plot_data = bank_df1.groupby(['balance_decile', 'loan_group'])['deposit'].apply(
    lambda x: (x == 'yes').mean()
).unstack()

plot_data.columns = ['대출 미보유','대출 보유'] # 아 맞다 언더바

In [ ]:
# 이 부분은 병주님이 분석하셨던 거 그대로 재현한겁니다. 
ax = sns.lineplot(data = plot_data, marker = 'o', linewidth=4, markersize=10, palette=['#5ca0de','#9dadbc'])

# plt.title('대출 유무와 잔고 증가량에 따른 예금 가입률', fontsize=16, y = 1.03)
plt.ylabel('')
plt.xlabel('')
plt.legend(fontsize = 16)
plt.grid(True, linestyle='--', alpha=0.5)
sns.despine()

plt.show()

In [ ]:
# 전환율, 비전환율 계산 
# 일단 그룹화
bank_df_balance = bank_df.copy()
bank_df_balance['loan_group'] = bank_df_balance['H/L'].apply(lambda x: f"대출 미보유" if x == 'HNLN' else '대출 보유') 
bank_df_depo = bank_df_balance.groupby(['loan_group','deposit']).size().unstack()

# 전환율, 비전환율 계산
bank_df_depo['sum'] = bank_df_depo['no'] + bank_df_depo['yes']
bank_df_depo['yes_rate'] = round((bank_df_depo['yes'] / bank_df_depo['sum']) * 100, 3)
bank_df_depo['no_rate'] = 100 - bank_df_depo['yes_rate']

# 및 칼럼명 변경
bank_df_depo.columns = ['가입하지 않음','가입','총계','전환율 (%)', '비전환율 (%)']
bank_df_depo

In [ ]:
plt.figure(figsize=(12, 6))

ax = sns.barplot(data= bank_df_depo, x='loan_group', y='전환율 (%)',legend=False, hue = 'loan_group', 
                palette=['#5ca0de', '#9dadbc'])

# 각 막대에 수치 추가
# for container in ax.containers:
#     ax.bar_label(container, fmt='%.1f%%', padding=3, fontsize=10, fontweight='bold')

plt.xlabel('')
plt.ylabel('')
# plt.title('고객 연령대별 예금 가입률', fontsize=16, y = 1.03)
plt.ylim(0, 85)
sns.despine()

plt.show()

### 정규성 검정

In [ ]:
# 데이터의 정규성 검정
# 정규성을 따름->T-test, 아님->맨-휘트니

group_yes_loan = plot_data['대출 미보유']
group_no_loan = plot_data['대출 보유']

# 1. 두 집단의 정규성 검정
stat_no, p_no = shapiro(group_no_loan)
stat_yes, p_yes = shapiro(group_yes_loan)

print(f"대출 없는 그룹 정규성 p-value: {p_no:.4e}")
print(f"대출 있는 그룹 정규성 p-value: {p_yes:.4e}")

# 2. 판정
if p_no < 0.05 or p_yes < 0.05:
    print("\n결과: 데이터가 정규성을 만족하지 않으므로 '맨-휘트니 U 검정'이 적절합니다.")
else:
    print("\n결과: 데이터가 정규성을 만족하므로 'T-test'를 고려할 수 있습니다.")

In [ ]:
# 등분산성 검정
stat, p = levene(group_yes_loan, group_no_loan)
print(p)

# 0.5->분산 같음! 

### 시각화(Q-Q plot)

#### 대출 없는 그룹

In [ ]:
probplot(group_no_loan, dist="norm", plot=plt)
plt.title("Q-Q Plot")
plt.show()


#### 대출 있는 그룹

In [ ]:
probplot(group_yes_loan, dist="norm", plot=plt)
plt.title("Q-Q Plot")
plt.show()


1. 샤피로-윌크 검정에서 p-value가 각각 0.6, 0.06이었음. 
2. 등분산성 검정에서 p-value가 0.5였음. (분산! 같다!)
3. **그러나** Q-Q plot을 그려보니 한쪽 그룹이 곡선이었고 (정규성을 띠지 않음)
4. **애초에 표본의 수가 너무 적어서(n-10) 맨-휘트니 U 검정을 하는 게 맞음.** 

In [ ]:
# 2. 맨-휘트니 U 검정 수행 (비모수 검정)
u_stat, p_val = stats.mannwhitneyu(group_no_loan, group_yes_loan, alternative='two-sided')

print("=== Mann-Whitney U Test 결과 ===")
print(f"U 통계량: {u_stat:.2f}")
print(f"p-value: {p_val:.4e}") # 지수 표기법으로 출력 (매우 작을 경우 대비)

# 3. 결과 해석
alpha = 0.05
if p_val < alpha:
    print(f"\n[해석] p-value가 {alpha}보다 작으므로, 대출 유무에 따른 두 집단의 CVR 차이는 통계적으로 매우 유의미합니다.")
    print(f"대출 없는 그룹 잔고 중앙값: {group_no_loan.median():.2f}")
    print(f"대출 있는 그룹 잔고 중앙값: {group_yes_loan.median():.2f}")
else:
    print(f"\n[해석] p-value가 {alpha}보다 크므로, 두 집단 간의 CVR 차이가 통계적으로 유의미하다고 볼 수 없습니다.")

# ANOVA
- 분산분석입니다. 단, 어노바는 '얘네들 중 다른 게 있다'만을 알려주기 때문에 귀무가설(다 또이또이여)이 기각되면 후속 분석을 헤야 합니다. 그 후속 분석 중 하나가 튜키(Tucky HSD)였던거고요. 
- 요약하자면: 
    1. 어노바: 환자들 그룹을 보니까 이들 중에 유전자의 발현량이 다른 그룹이 있었다
    2. 튜키: 흡연자랑 비흡연자랑 발현량이 다르네? 
## 연령대, Contact 횟수별 ANOVA (2-way)
- 요인이 두 개라서 이원분석이지요. (1-way: 요인이 하나)

In [ ]:
bank_df2 = bank_df.query('20 <= age_group <= 70')

# 1. 그래프와 동일한 집계 데이터(CVR) 생성
def bin_campaign(c):
    if c == 1: return '1'
    elif c == 2: return '2'
    elif c == 3: return '3'
    elif 4 <= c <= 5: return '4-5'
    elif 6 <= c <= 10: return '6-10'
    else: return '11+'

# bank_df1 (plus 잔고 그룹) 기반으로 집계
bank_df2['campaign_bin'] = bank_df2['campaign'].apply(bin_campaign)

# 연령대별, 캠페인 횟수별 CVR 계산
cvr_table = bank_df2.groupby(['age_group', 'campaign_bin'])['deposit'].apply(
    lambda x: (x == 'yes').mean() * 100
).reset_index(name='cvr')
order = ['1', '2', '3', '4-5', '6-10', '11+']
cvr_table['campaign_bin'] = pd.Categorical(cvr_table['campaign_bin'], categories=order, ordered=True)
cvr_table = cvr_table.sort_values(['age_group', 'campaign_bin'])

display(cvr_table)

# 2. ANOVA 수행 (상호작용 항을 제외한 Additive Model)
# 데이터 포인트가 적을 때는 '*' 대신 '+'를 사용해야 분산 계산이 가능합니다.
# "CVR이 연령대에 따라 다른가? 캠페인 횟수에 따라 다른가?"를 검정합니다.
model = ols('cvr ~ C(age_group) + C(campaign_bin)', data=cvr_table).fit()
anova_results = sm.stats.anova_lm(model, typ=2)

print("=== CVR(가입률) 기반 Two-way ANOVA 결과 ===")
print(anova_results.round(2))

In [ ]:
# 캠페인으로 일원분석
model2 = ols('cvr ~ C(campaign_bin)', data=cvr_table).fit()
anova_results2 = sm.stats.anova_lm(model2)

print("=== CVR(가입률) 기반 Two-way ANOVA 결과 ===")
print(anova_results2.round(2))

## 왜 2-way?
- 일원분석 하면 빈도로만 했을때는 연령대가 오히려 노이즈가 돼서 잘못된 결론을 내놓더라... 
- 나이로 2-way ANOVA를 하니까 원인이 밝혀졌다

### ANOVA 친구 튜키
- 설명은 위에서 했으니까 생략... 튜기가 있다는 건 얘네들 중 다른 게 있다는 게 어노바에서 밝혀진겁니다. 
- reject=True인 부분이 이놈이 다르다! 입니다. 

In [ ]:
# 1. 연령대(age_group)에 대한 사후 검정
print("=== 연령대별 CVR 차이 (Tukey HSD) ===")
tukey_age = pairwise_tukeyhsd(endog=cvr_table['cvr'], groups=cvr_table['age_group'], alpha=0.05)
print(tukey_age)

# 2. 캠페인 횟수(campaign_bin)에 대한 사후 검정
print("\n=== 캠페인 횟수별 CVR 차이 (Tukey HSD) ===")
# 횟수는 우리가 정한 순서대로 비교하기 위해 정렬된 데이터를 사용합니다.
tukey_campaign = pairwise_tukeyhsd(endog=cvr_table['cvr'], groups=cvr_table['campaign_bin'], alpha=0.05)
print(tukey_campaign)

### 시각화

In [ ]:
# 연령대 튜키 결과 시각화
fig = tukey_age.plot_simultaneous(comparison_name=70, figsize=(10, 6))
plt.title("Tukey HSD: Age Group Comparison (Standard: Age 70)")
plt.xlabel("CVR (%) Difference")
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(x='campaign_bin', y='cvr', data=cvr_table, palette='Blues')
sns.swarmplot(x='campaign_bin', y='cvr', data=cvr_table, color=".25") # 실제 CVR 점들 추가
plt.title("CVR Distribution by Campaign Contact Frequency")
plt.ylabel("CVR (%)")
plt.xlabel("Campaign Contact Frequency (bin)")
plt.show()

- 연령대에 따른 가입률 차이는 존재하지만, 접촉 횟수가 늘어나면 가입률이 떨어지는 양상은 똑같습니다. 
- 그래서 접촉 횟수에 따른 가입 양상에서 올 False였던 겁니다. 

# 상관분서억

- 아. 쓰이지는 않았습니다. 

## 스피어만 상관분석(대출 유무와 deposit)
- deposit이 범주형 데이터인데, 범주형 데이터에는 스피어만 상관분석을 합니다. 

In [ ]:
# 대출 그룹에 따른 deposit
corr_data = bank_df1[['loan_group','deposit']].copy()

# 범주형 데이터 더미화
# 대출 없음: 1, 있음: 0
# 가입 예스: 1, 노: 0
corr_data['loan_dummy'] = corr_data['loan_group'].apply(lambda x: 1 if x == 'HNLN' else 0)
corr_data['deposit_dummy'] = corr_data['deposit'].apply(lambda x: 1 if x == 'yes' else 0)

# 스피어만 상관계수 분석
spearman_corr_matrix = corr_data[['loan_dummy','deposit_dummy']].corr(method='spearman')
display(spearman_corr_matrix.round(2))

- 대출유무와 가입률간에는 약한 양의 상관관계가 있다. 

## 피어슨 상관계수(잔고 분위별 CVR)

In [ ]:
# 잔고가 +인 사람
bank_df1 = bank_df.query('balance_group == "plus"').copy()

def define_loan_group(row):
    if row['housing'] == 'no' and row['loan'] == 'no':
        return 'HNLN'
    else:
        return 'Has_Loan'

bank_df1['loan_group'] = bank_df1.apply(define_loan_group, axis=1)

bank_df1['balance_decile'] = pd.qcut(bank_df1['balance'], 10, labels=False)
# 그룹별 가입률 계산
plot_data = bank_df1.groupby(['balance_decile', 'loan_group'])['deposit'].apply(
    lambda x: (x == 'yes').mean()
).unstack()

plot_data.columns = ['대출 미보유','대출 보유']

plot_data

In [ ]:
# 잔고 레벨이 인덱스로 빠져서... 
plot_data_reset = plot_data.reset_index()

plot_data_corr = plot_data_reset.corr()
display(plot_data_corr.round(2))

- 잔고등급에 따른 대출 미보유자의 상관계수: 0.85
- 잔고 등급에 따른 대출 보유자의 상관계수: 0.96
> 부채가 있는 고객일수록 가용 자산의 규모에 따라 가입 의사결정을 보수적이고 규칙적으로 내리고 있음

# MI(Mutual Information)
- 다른 분이 하신건데, 저도 걍 시험삼아 해보려고요. 생물정보학에서도 많이 쓰는거라.. 

## 범주화 후 분석

In [ ]:
MI_df = bank_df.copy()

In [ ]:
y = MI_df['deposit'].map({'yes': 1, 'no': 0}) # 가입 여부
X = MI_df.drop(columns=['deposit']) # 빼고 다

X_encoded = pd.get_dummies(X, drop_first=True) # 일단 더미 만듭니다
discrete_indices = [i for i, col in enumerate(X_encoded.columns) if X_encoded[col].nunique() <= 2] # 더미화 했기 떄문에 0 아니면 1입니다. (범주형)

mi_scores = mutual_info_classif(X_encoded, y, discrete_features=discrete_indices, random_state=42)
mi_results = pd.Series(mi_scores, index=X_encoded.columns).sort_values(ascending=False)

print("--- [전체 변수] MI 점수 결과 (상위 20개) ---")
print(mi_results.head(20))

## 별도의 범주화 없이 진행

In [ ]:
MI_final = bank_df.copy()
le = LabelEncoder()
for col in MI_final.select_dtypes(include=['object']).columns:
    MI_final[col] = le.fit_transform(MI_final[col])

In [ ]:
X_all = MI_final.drop(columns=['deposit'])
y_all = MI_final['deposit']

mi_scores = mutual_info_classif(X_all, y_all, discrete_features='auto', random_state=42)

result_df = pd.DataFrame({
    'Feature': X_all.columns,
    'MI_Score': mi_scores
})

result_df = result_df.sort_values(by='MI_Score', ascending=False).reset_index(drop=True)

print("--- [정렬 완료] 통합형 MI 결과 상위 20개 ---")
print(result_df.head(20))

- 그래! 얘가 되는데 쟤가 안 될리가 없지! 

# FAMD
- Factor analysis of mixed data
- 이 데이터로 PCA(주성분분석)를 할 수'는' 있어요. 근데 위에서 서술했지만 수치형만 '빼서' 해야됩니다. 범주형 일일이 더미화하면... 저기 아래 더미데이터 가 보시면 깨달음을 얻으실 수 있습니다. 
- 그래서 혼합형으롣 할 수 있는 FAMD를 할거예요. 

In [ ]:
bank_FAMD = bank_df.copy()

# 결과와 원인'들'로 분리
y = bank_FAMD['deposit'] 
X = bank_FAMD.drop(columns=['deposit'])

In [ ]:
num_cols = [
    'age', 'balance', 'day', 'duration',
    'campaign', 'pdays', 'previous'
]

cat_cols = [
    'job', 'marital', 'education', 'default',
    'housing', 'loan', 'contact', 'month', 'poutcome'
]

X_famd = bank_FAMD[num_cols + cat_cols].copy()

# 🔑 핵심
X_famd[num_cols] = X_famd[num_cols].astype(float)
X_famd[cat_cols] = X_famd[cat_cols].astype('category')

In [ ]:
famd = prince.FAMD(
    n_components=2, 
    random_state=42
)

X_famd_transformed = famd.fit_transform(X_famd)

In [ ]:
# 샘플 좌표
print(X_famd_transformed.head())

## 설명력
- 위: 퍼센테이지/아래: 설명력

In [ ]:
famd.percentage_of_variance_ # 차원의 설명력 (%)

In [ ]:
famd.eigenvalues_ # 그 뭐라고 해야되나... 원점수? 

In [ ]:
# 0번 차원 기여도 시각화
famd.column_contributions_[0].sort_values().plot(kind='barh', figsize=(8, 6), color='#00498c')
plt.title('Component 0에 대한 변수 기여도')
plt.xlabel('Contribution (%)')
plt.show()

In [ ]:
# 1번 차원 기여도 시각화
famd.column_contributions_[1].sort_values().plot(kind='barh', figsize=(8, 6), color='#00498c')
plt.title('Component 1에 대한 변수 기여도')
plt.xlabel('Contribution (%)')
plt.show()

In [ ]:
summary = famd.eigenvalues_summary.copy()

# 퍼센트 기호(%)가 포함된 문자열을 숫자로 변환하는 함수
def clean_pct(x):
    if isinstance(x, str):
        return float(x.replace('%', ''))
    return x

summary['% of variance'] = summary['% of variance'].apply(clean_pct)
summary['% of variance (cumulative)'] = summary['% of variance (cumulative)'].apply(clean_pct)

# 3. PCA 스타일 성적표 출력
print("--- FAMD 결과 성적표 ---")
for i, row in summary.iterrows():
    # 1.0 기준 소수점으로 변환 (보내주신 PCA 예시 포맷)
    variance_ratio = row['% of variance'] / 100
    cumulative_variance = row['% of variance (cumulative)'] / 100
    
    print(f"제 {int(i)+1} 주성분: {variance_ratio:.4f} (누적: {cumulative_variance:.4f})")

# 4. 시각화 (Scree Plot)
plt.figure(figsize=(10, 5))
plt.bar(summary.index, summary['% of variance'], color='skyblue', label='개별 설명력')
plt.plot(summary.index, summary['% of variance (cumulative)'], color='orange', marker='o', label='누적 설명력')
plt.title('FAMD 차원별 설명력 성적표')
plt.xlabel('주성분')
plt.ylabel('설명력 (%)')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.show()

## 관측치 좌표

In [ ]:
famd.row_coordinates(X_famd).head()

## 변수 기여도

In [ ]:
df_comp0 = famd.column_contributions_.sort_values(by=0, ascending=False)

df_comp0

In [ ]:
df_comp1 = famd.column_contributions_.sort_values(by=1, ascending=False)

df_comp1

- 공부할게 늘어서 골치아프네요... 
- 위에 다른건 차치하고, 변수 기여도를 봐주세요. 다른것까지 끼면 골치 아프니까 일단 여기만 봅시다. 나머지는 추가로 알아보고 업데이트 하겠습니다. 

>왜 표가 두 개죠?
- 표가 두개인 이유는 간단합니다. 0차원... 그러니까 0번 칼럼은 은행의 마케팅 측면에서 이 변수들이 얼마나 기여하고 있는가이고 1차원... 1번 칼럼은 고객의 특성 측면에서 이 변수들이 얼마나 기여하고 있는가입니다. df_comp0은 은행의 마케팅 차원인데, 여기서 가장 많이 기여하고 있는 TOP 3가 월(month), 접촉 수단, 직업이라는 얘기입니다. 
- 여기서 조심하셔야 할 게, 아니 그럼 이것들로 분석을 했어야 하는 것 아닌가요? 라고 생각하시면 안 됩니다. 그건 위에 있는 MI에서 확인한거예요. FAMD는 주성분분석의 친구친구라서 주성분분석처럼 차원 축소를 하되, PCA처럼 수치형만 편식하는 게 아니라(MCA는 범주형만...) 골고루 다 먹는 애예요. 그러니까 **음 이렇군**만 보시면 되겠습니다. 

## 시각화

In [ ]:
plt.scatter(
    X_famd_transformed[0],
    X_famd_transformed[1],
    c=bank_FAMD['deposit'].map({'yes':1, 'no':0}),
    s=5,
    alpha=0.5, 
    cmap='viridis')
plt.colorbar(label='Subscription')
plt.show()

- Blues 안보여서 viridis 한건데도 안보이는거 실화냐... 
- 이거는 분석이 망한 게 아니라, 이 데이터가 고차원적이라서 그런겁니다. 이것도 걍 가입한 사람과 아닌 사람이 이런 차이가 있었다 정도로 보시면 되겠습니다. 
- 축 저게 최선이었냐고... 

# 더미데이터

# PCA(주성분분석)

- MI는 다른 분이 하셨고... 이친구는 MI랑 다릅니다. 얘는 쉽게 말하자면 피카츄 라이츄 피츄 데덴네 빠모 에몽가를 전기쥐로 줄이는 작업입니다. 
- 근데 이 방법은 수치형에는 통하는데 범주형 섞여있으면 안 통하니까 착한 분석가 여러분들은 수치형만 뽑아서 하시거나 다른 방법을 물어보세요. 
- 이래서 사람은 공부를 해야 합니다. 

In [ ]:
# 사전 더미화 (월 일 제외)
# 이거 안햅주면 오류납니다... 
df_encoded = pd.get_dummies(bank_df, columns=['job', 'marital', 'education', 'default','contact', 'H/L','poutcome', 'previous_group', 'balance_group', 'campaign_group'])

In [ ]:
# 스탠다드 스케일러
# 너무 확 튀면 그것떄문에 결과 왜곡되니까 잡아주는거예요. 
scaler = StandardScaler()
numeric_scale = ['duration','balance','pdays','age']

df_encoded[numeric_scale] = scaler.fit_transform(bank_df[numeric_scale])

In [ ]:
# 칼럼 다이어트(안쓸거 빼고 들어갑니다)
cols_to_drop = ['campaign','pdays','previous','age_group','group_category','deposit', 'month', 'day', 'job', 'marital', 'education', 'housing','loan','default','contact', 'H/L','poutcome', 'previous_group', 'balance_group', 'campaign_group']
existing_cols = [c for c in cols_to_drop if c in df_encoded.columns]
X = df_encoded.drop(existing_cols, axis=1, inplace=True)

In [ ]:
df_encoded

In [ ]:
# 숫자가 아닌(문자열인) 컬럼들만 쏙 골라내기
non_numeric_cols = df_encoded.select_dtypes(exclude=[np.number]).columns.tolist()

print("범인 검거 리스트:", non_numeric_cols) # 다 부울이라 괜찮습니당

## 사전작업 끝났으면 들어가주는 게 인지상정!

In [ ]:
pca = PCA(n_components=0.90)
X_pca = pca.fit_transform(df_encoded)

In [ ]:
print("성공! 최종 변수 개수:", df_encoded.shape[1])
print("압축된 주성분 개수:", X_pca.shape[1])

In [ ]:
# 1. 각 주성분의 설명력(분산 비율) 확인
explained_variance = pca.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)

print("--- PCA 결과 성적표 ---")
for i, ratio in enumerate(explained_variance):
    print(f"제 {i+1} 주성분: {ratio:.4f} (누적: {cumulative_variance[i]:.4f})")

# 2. 제1주성분(PC1)에 가장 큰 영향을 준 변수 top 10
# (주성분이 어떤 원래 변수들로 만들어졌는지 확인)
loadings = pd.DataFrame(pca.components_[0], index=df_encoded.columns, columns=['PC1_Loading'])
print("\n--- PC1을 구성하는 핵심 변수 TOP 10 ---")
print(loadings['PC1_Loading'].abs().sort_values(ascending=False).head(10))

- 이 데이터가 설명력을 85% 이상 갖기 위해서는 제 13 주성분까지 필요하다. 네, 이거 실화입니다. 
- 제 16 주성분까지 다 있어야 설명력을 90% 이상 갖는다. 
>고객의 전환율에 영향을 끼치는 요인이 되게 다각적임

- 근데 생각해보니 저거 더미화해서 주성분이 되게... 세분류가 됐는데... 이건 다른 방법 없는지 함 찾아보겠습니다. \
제미나이가 해준거긴 한데, 범주화 저렇게 하면 안될 것 같은데... 

## Scree plot (PCA 시각화)

In [ ]:
# 데이터 준비 (가장 기여도가 높은 변수명을 라벨로 사용)
pc_labels = []
for i in range(len(pca.components_)):
    top_feature = df_encoded.columns[np.argmax(np.abs(pca.components_[i]))]
    pc_labels.append(f"PC{i+1}\n({top_feature})")

exp_var_pca = pca.explained_variance_ratio_

plt.figure(figsize=(16, 8))
bars = plt.bar(pc_labels, exp_var_pca, color='skyblue', alpha=0.7)

# 값 표시
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 0.005, f'{yval:.2f}', ha='center', va='bottom')

plt.title('주성분별 설명력 및 대표 변수 (Scree Plot)', fontsize=15)
plt.ylabel('설명 가능한 분산 비율')
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## Loading plot

In [ ]:
# 로딩 값 추출
loadings = pca.components_.T * np.sqrt(pca.explained_variance_)
loading_df = pd.DataFrame(loadings, index=df_encoded.columns, columns=[f'PC{i+1}' for i in range(len(pca.components_))])

plt.figure(figsize=(10, 8))
plt.axhline(0, color='black', linewidth=1)
plt.axvline(0, color='black', linewidth=1)

# 주요 변수들만 화살표로 표시 (너무 많으면 복잡하므로 상위 기여 변수 위주)
top_features = loading_df.abs().sum(axis=1).sort_values(ascending=False).head(15).index

for feature in top_features:
    plt.arrow(0, 0, loading_df.loc[feature, 'PC1'], loading_df.loc[feature, 'PC2'], 
              color='r', alpha=0.5, head_width=0.02)
    plt.text(loading_df.loc[feature, 'PC1']*1.15, loading_df.loc[feature, 'PC2']*1.15, 
             feature, color='g', ha='center', va='center', fontsize=10)

plt.xlabel('PC1 (나이/자산 축)')
plt.ylabel('PC2 (결혼/대출 축)')
plt.title('변수별 로딩 플롯 (PC1 vs PC2)', fontsize=15)
plt.grid(alpha=0.3)
plt.xlim(-1, 1)
plt.ylim(-1, 1)
plt.tight_layout()
plt.show()

- 악 내눈... 지극히 정상입니다. 
- 저기서 자기주장을 하는 순서가 나이, 잔고, 접촉 시간이라는 얘기... 

### 결론
- 윤성님의 MI 결과를 보니, 우리 칼럼 잘 고른 거 맞음. 
- 이 데이터는 85%의 설명력을 확보하기 위해 최소 13개의 주성분이 필요하다. 